# Department of Labor appreticeship program data

This short notebook analyzes data from the DoL "Apprenticeship Participant Characteristics and Training Outcomes" data, available here: https://data.dol.gov/datasets/10264 

Below, we use the polars module on the 3 GB zip file you can download from that webpage to analyze this data. Then we show how to grab data via the API.

# Analyzing downloaded data

In [1]:
import polars as pl
import zipfile
import pandas as pd
import pyarrow
import json

zip_path = "ETA_apprenticeship_data.zip"

with zipfile.ZipFile(zip_path) as z:
    df = pl.concat([
        pl.read_csv(
            z.open(f"Apprenticeship (active, new, exiter)_chunk_{i}.csv"),
            infer_schema_length=0,  # reads all columns as strings
            ignore_errors=True      # skips unparseable rows
        )
        for i in range(1, 176)
    ])

print(f"This data set has {df.shape[0]:,} rows and {df.shape[1]} columns.")
print("Here is a preview of some of the data:")
print(df.head())

This data set has 8,745,442 rows and 34 columns.
Here is a preview of some of the data:
shape: (5, 34)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ Fiscal    ┆ Table     ┆ Start     ┆ Exit Date ┆ … ┆ APPR_COUN ┆ APPR_COUN ┆ STARTING_ ┆ Exit_Wag │
│ Year      ┆ Name      ┆ Date      ┆ ---       ┆   ┆ TY_FIPS   ┆ TY_NAME   ┆ WAGE      ┆ e        │
│ ---       ┆ ---       ┆ ---       ┆ str       ┆   ┆ ---       ┆ ---       ┆ ---       ┆ ---      │
│ str       ┆ str       ┆ str       ┆           ┆   ┆ str       ┆ str       ┆ str       ┆ str      │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
│ 2014      ┆ V_OA_APPR ┆ 09 02     ┆ 2015-05-2 ┆ … ┆ 28099     ┆ Neshoba   ┆ 8.37      ┆ 7.65     │
│           ┆ ENTICE_AC ┆ 2010      ┆ 7         ┆   ┆           ┆ County    ┆           ┆          │
│           ┆ TIVE_V1   ┆ 00:00:00  ┆           ┆   ┆           ┆           ┆           ┆

In [2]:
# ── 2. CAST COLUMNS TO CORRECT TYPES ─────────────────────────────────────────
 
categorical_cols = [
    "Fiscal Year", "Table Name", "Program View", "NAICS_CD", "Age Cohort",
    "Ethnicity", "Gender", "Race", "Veteran Status Title",
    "Individuals with Disabilities", "Industry", "Apprentice Status Code",
    "HUD_State", "County Fips", "County Name", "State", "APPR_STATE",
    "APPR_COUNTY_FIPS", "APPR_COUNTY_NAME", "Education",
]
 
boolean_cols = ["UNION_Y_N", "ACTIVE_ADJUST", "COMPLETER_ADJUST", "NEW_ADJUST"]
 
numeric_cols = ["APPRENTICE_ID", "EXIT_WAGE_ADJUST", "START_WAGE_ADJUST",
                "STARTING_WAGE", "Exit_Wage"]
 
date_cols = ["Start Date", "Exit Date"]
 
# Text cols (Apprentice Number, Program Number, Occupation) stay as strings
 
df = df.with_columns(
    # Categorical
    [pl.col(c).cast(pl.Categorical) for c in categorical_cols if c in df.columns]
    # Numeric
    + [pl.col(c).cast(pl.Float64, strict=False) for c in numeric_cols if c in df.columns]
    # Boolean (stored as "0"/"1" strings)
    + [pl.col(c).cast(pl.Int8, strict=False).cast(pl.Boolean) for c in boolean_cols if c in df.columns]
    # Dates — different formats for each column
    + [pl.col("Start Date").str.to_date(format="%m %d %Y %H:%M:%S", strict=False)]
    + [pl.col("Exit Date").str.to_date(format="%m/%d/%Y", strict=False)]
)
 
print("Schema after casting:")
print(df.schema, "\n")

Schema after casting:
Schema({'Fiscal Year': Categorical, 'Table Name': Categorical, 'Start Date': Date, 'Exit Date': Date, 'Apprentice Number': String, 'Program Number': String, 'Program View': Categorical, 'NAICS_CD': Categorical, 'UNION_Y_N': Boolean, 'APPRENTICE_ID': Float64, 'Age Cohort': Categorical, 'Ethnicity': Categorical, 'Gender': Categorical, 'Race': Categorical, 'Veteran Status Title': Categorical, 'Individuals with Disabilities': Categorical, 'Industry': Categorical, 'Occupation': String, 'Education': Categorical, 'Apprentice Status Code': Categorical, 'ACTIVE_ADJUST': Boolean, 'COMPLETER_ADJUST': Boolean, 'EXIT_WAGE_ADJUST': Float64, 'NEW_ADJUST': Boolean, 'START_WAGE_ADJUST': Float64, 'HUD_State': Categorical, 'County Fips': Categorical, 'County Name': Categorical, 'State': Categorical, 'APPR_STATE': Categorical, 'APPR_COUNTY_FIPS': Categorical, 'APPR_COUNTY_NAME': Categorical, 'STARTING_WAGE': Float64, 'Exit_Wage': Float64}) 



In [3]:
quick_view = df.head()
quick_view

Fiscal Year,Table Name,Start Date,Exit Date,Apprentice Number,Program Number,Program View,NAICS_CD,UNION_Y_N,APPRENTICE_ID,Age Cohort,Ethnicity,Gender,Race,Veteran Status Title,Individuals with Disabilities,Industry,Occupation,Education,Apprentice Status Code,ACTIVE_ADJUST,COMPLETER_ADJUST,EXIT_WAGE_ADJUST,NEW_ADJUST,START_WAGE_ADJUST,HUD_State,County Fips,County Name,State,APPR_STATE,APPR_COUNTY_FIPS,APPR_COUNTY_NAME,STARTING_WAGE,Exit_Wage
cat,cat,date,date,str,str,cat,cat,bool,f64,cat,cat,cat,cat,cat,cat,cat,str,cat,cat,bool,bool,f64,bool,f64,cat,cat,cat,cat,cat,cat,cat,f64,f64
"""2014""","""V_OA_APPRENTICE_ACTIVE_V1""",2010-09-02,null,"""MS10N007699""","""MS021040006""","""State""","""236220""",null,1.402604e6,"""Ages 24 and Under""","""Non-Hispanic or Latino""","""Male""","""White""","""Non Veteran""","""Participant Did Not Self-Ident…","""Construction""","""ELECTRICIAN (Alternate Title: …","""High School graduate (includin…","""CA""",null,null,null,null,8.37,"""MS""","""28089""","""Madison County""","""MS""","""MS""","""28099""","""Neshoba County""",8.37,7.65
"""2014""","""V_OA_APPRENTICE_ACTIVE_V1""",2010-08-30,null,"""PA10N043481""","""ZA002530001""","""National""","""238220""",null,1.405965e6,"""Ages 24 and Under""","""Non-Hispanic or Latino""","""Male""","""White""","""Non Veteran""","""Participant Did Not Self-Ident…","""Construction""","""PIPE FITTER (Construction)""","""High School graduate (includin…","""CA""",null,null,null,null,16.68,"""MD""","""24027""","""Howard County""","""ZA""","""PA""","""42041""","""Cumberland County""",16.68,29.21
"""2014""","""V_OA_APPRENTICE_ACTIVE_V1""",2010-05-11,null,"""TX10N045768""","""TX011410002""","""State""","""238210""",null,1.406142e6,"""Ages 25-54""","""Hispanic or Latino""","""Male""","""Participant Did Not Self-Ident…","""Non Veteran""","""Participant Did Not Self-Ident…","""Construction""","""ELECTRICIAN (Alternate Title: …","""High School graduate (includin…","""CO""",null,null,null,null,14.02,"""TX""","""48167""","""Galveston County""","""TX""","""TX""","""48201""","""Harris County""",14.02,21.2
"""2014""","""V_OA_APPRENTICE_ACTIVE_V1""",2010-10-21,null,"""ND09N001912""","""MO012620001""","""State""","""221122""",null,1.408069e6,"""Ages 24 and Under""","""Non-Hispanic or Latino""","""Male""","""White""","""Non Veteran""","""Participant Did Not Self-Ident…","""Utilities""","""LINE MAINTAINER (Alternate Tit…","""Some College or Associate's de…","""CO""",null,null,null,null,24.68,"""IA""","""19181""","""Warren County""","""IA""","""ND""","""38101""","""Ward County""",24.68,38.25
"""2014""","""V_OA_APPRENTICE_ACTIVE_V1""",2010-11-06,null,"""WI10N001257""","""MO012620001""","""State""","""221122""",null,1.411987e6,"""Ages 25-54""","""Non-Hispanic or Latino""","""Male""","""White""","""Non Veteran""","""Participant Did Not Self-Ident…","""Utilities""","""LINE MAINTAINER (Alternate Tit…","""High School graduate (includin…","""CO""",null,null,null,null,25.18,"""IA""","""19181""","""Warren County""","""IA""","""WI""","""55121""","""Trempealeau County""",25.18,42.14


In [4]:
# ── 3. MISSINGNESS ANALYSIS ───────────────────────────────────────────────────
 
total_rows = df.shape[0]
 
missing = (
    df.null_count()
      .unpivot(variable_name="column", value_name="null_count")
      .with_columns(
          (pl.col("null_count") / total_rows * 100).round(2).alias("pct_missing")
      )
      .sort("pct_missing", descending=True)
)
 
print("=" * 55)
print("MISSINGNESS BY COLUMN")
print("=" * 55)
 
# Format with commas
missing_pd = missing.to_pandas()
missing_pd["null_count"] = missing_pd["null_count"].apply(lambda x: f"{x:,}")
missing_pd["pct_missing"] = missing_pd["pct_missing"].apply(lambda x: f"{x:.2f}%")
print(missing_pd.to_string(index=False))
 
complete_cols = missing.filter(pl.col("null_count") == 0)["column"].to_list()
print(f"\nColumns with NO missing data ({len(complete_cols)}): {complete_cols}")
 
high_missing = missing.filter(pl.col("pct_missing") > 50)
if len(high_missing):
    print(f"\nColumns missing >50% of values:")
    print(high_missing)

MISSINGNESS BY COLUMN
                       column null_count pct_missing
                    Exit Date  8,745,442     100.00%
                    UNION_Y_N  8,745,442     100.00%
                ACTIVE_ADJUST  8,745,442     100.00%
             COMPLETER_ADJUST  8,745,442     100.00%
                   NEW_ADJUST  8,745,442     100.00%
             EXIT_WAGE_ADJUST  7,942,999      90.82%
                    Exit_Wage  3,978,770      45.50%
                STARTING_WAGE  2,682,195      30.67%
                     NAICS_CD    930,994      10.65%
                   Start Date    802,443       9.18%
            START_WAGE_ADJUST    802,443       9.18%
                   APPR_STATE     27,348       0.31%
             APPR_COUNTY_FIPS     27,348       0.31%
             APPR_COUNTY_NAME     27,348       0.31%
                  County Fips     20,510       0.23%
                  County Name     20,510       0.23%
                    HUD_State     16,564       0.19%
                  Fiscal

In [5]:
# ── 4. SUMMARY STATISTICS ─────────────────────────────────────────────────────
 
print("\n" + "=" * 55)
print("NUMERIC COLUMN SUMMARY STATS")
print("=" * 55)
 
numeric_summary = df.select(
    [c for c in numeric_cols if c in df.columns]
).describe().to_pandas()
 
# Format numeric values with commas, preserving the "statistic" label column
def fmt(val):
    try:
        f = float(val)
        return f"{f:,.2f}"
    except (ValueError, TypeError):
        return val
 
value_cols = [c for c in numeric_summary.columns if c != "statistic"]
numeric_summary[value_cols] = numeric_summary[value_cols].map(fmt)
print(numeric_summary.to_string(index=False))
 
print("\n" + "=" * 55)
print("CATEGORICAL COLUMN VALUE COUNTS (top 10 each)")
print("=" * 55)
for col in categorical_cols:
    if col not in df.columns:
        continue
    print(f"\n--- {col} ---")
    vc = df[col].value_counts(sort=True).head(10).to_pandas()
    vc["count"] = vc["count"].apply(lambda x: f"{x:,}")
    print(vc.to_string(index=False))
 
print("\n" + "=" * 55)
print("DATE COLUMN RANGES")
print("=" * 55)
for col in date_cols:
    if col not in df.columns:
        continue
    print(f"\n{col}:")
    print(f"  Min: {df[col].min()}")
    print(f"  Max: {df[col].max()}")
    print(f"  Nulls: {df[col].null_count():,} ({df[col].null_count()/total_rows*100:.1f}%)")
 
print("\n" + "=" * 55)
print("BOOLEAN COLUMN SUMMARY")
print("=" * 55)
for col in boolean_cols:
    if col not in df.columns:
        continue
    print(f"\n{col}:")
    vc = df[col].value_counts(sort=True).to_pandas()
    vc["count"] = vc["count"].apply(lambda x: f"{x:,}")
    print(vc.to_string(index=False))


NUMERIC COLUMN SUMMARY STATS
 statistic APPRENTICE_ID EXIT_WAGE_ADJUST START_WAGE_ADJUST STARTING_WAGE     Exit_Wage
     count  8,745,442.00       802,443.00      7,942,999.00  6,063,247.00  4,766,672.00
null_count          0.00     7,942,999.00        802,443.00  2,682,195.00  3,978,770.00
      mean  3,035,612.79            20.54             12.70         43.12        444.51
       std  1,111,503.41            17.26              9.62     14,117.83     79,650.54
       min          2.00             0.00              0.00         -0.01          0.00
       25%  2,084,449.00             0.00              0.00         13.05         16.00
       50%  2,762,670.00            22.00             14.61         16.50         22.00
       75%  4,023,549.00            33.17             18.91         20.44         32.85
       max  4,980,071.00           135.00            150.00 12,132,017.00 87,906,523.00

CATEGORICAL COLUMN VALUE COUNTS (top 10 each)

--- Fiscal Year ---
Fiscal Year     count


In [6]:
# ── FILTER TO COMPLETED + CANCELLED ──────────────────────────────────────────

exiter_df = df.filter(
    pl.col("Apprentice Status Code").cast(pl.Utf8).is_in(["CO", "CA"])
).with_columns(
    (pl.col("Apprentice Status Code").cast(pl.Utf8) == "CO").alias("is_completed")
)
print(f"Completed + cancelled rows: {exiter_df.shape[0]:,} (filtered from {df.shape[0]:,} total)\n")


# ── 1. TOP 10 OCCUPATIONS BY PERCENTAGE ──────────────────────────────────────

print("=" * 55)
print("TOP 10 OCCUPATIONS")
print("=" * 55)

total = df.shape[0]

top_occupations = (
    df.group_by("Occupation")
    .agg(pl.len().alias("count"))
    .sort("count", descending=True)
    .head(10)
    .with_columns(
        (pl.col("count") / total * 100).round(2).alias("pct")
    )
)

occ_pd = top_occupations.to_pandas()
occ_pd["count"] = occ_pd["count"].apply(lambda x: f"{x:,}")
occ_pd["pct"] = occ_pd["pct"].apply(lambda x: f"{x:.2f}%")
occ_pd.index = range(1, 11)
print(occ_pd.to_string())


# ── 2. COMPLETION RATE BY STARTING WAGE QUARTILE ─────────────────────────────

print("\n" + "=" * 55)
print("COMPLETION RATE BY STARTING WAGE QUARTILE")
print("=" * 55)
print("(only apprentices with a recorded starting wage)\n")

wage_df = exiter_df.filter(pl.col("STARTING_WAGE").is_not_null())

q25, q50, q75 = (
    wage_df["STARTING_WAGE"].quantile(0.25),
    wage_df["STARTING_WAGE"].quantile(0.50),
    wage_df["STARTING_WAGE"].quantile(0.75),
)

wage_df = wage_df.with_columns(
    pl.when(pl.col("STARTING_WAGE") <= q25)
      .then(pl.lit(f"Q1 (≤ ${q25:,.2f})"))
      .when(pl.col("STARTING_WAGE") <= q50)
      .then(pl.lit(f"Q2 (${q25:,.2f}–${q50:,.2f})"))
      .when(pl.col("STARTING_WAGE") <= q75)
      .then(pl.lit(f"Q3 (${q50:,.2f}–${q75:,.2f})"))
      .otherwise(pl.lit(f"Q4 (> ${q75:,.2f})"))
      .alias("wage_quartile")
)

completion_by_wage = (
    wage_df.group_by("wage_quartile")
    .agg([
        pl.len().alias("total"),
        pl.col("is_completed").sum().alias("completed"),
    ])
    .with_columns(
        (pl.col("completed") / pl.col("total") * 100).round(2).alias("completion_rate_pct")
    )
    .sort("wage_quartile")
)

wage_pd = completion_by_wage.to_pandas()
wage_pd["total"] = wage_pd["total"].apply(lambda x: f"{x:,}")
wage_pd["completed"] = wage_pd["completed"].apply(lambda x: f"{x:,}")
wage_pd["completion_rate_pct"] = wage_pd["completion_rate_pct"].apply(lambda x: f"{x:.2f}%")
print(wage_pd.to_string(index=False))


# ── 3. COMPLETION RATE BY FISCAL YEAR ────────────────────────────────────────

print("\n" + "=" * 55)
print("COMPLETION RATE BY FISCAL YEAR")
print("=" * 55)

completion_by_year = (
    exiter_df.filter(pl.col("Fiscal Year").is_not_null())
    .group_by("Fiscal Year")
    .agg([
        pl.len().alias("total"),
        pl.col("is_completed").sum().alias("completed"),
    ])
    .with_columns(
        (pl.col("completed") / pl.col("total") * 100).round(2).alias("completion_rate_pct")
    )
    .sort("Fiscal Year")
)

year_pd = completion_by_year.to_pandas()
year_pd["total"] = year_pd["total"].apply(lambda x: f"{x:,}")
year_pd["completed"] = year_pd["completed"].apply(lambda x: f"{x:,}")
year_pd["completion_rate_pct"] = year_pd["completion_rate_pct"].apply(lambda x: f"{x:.2f}%")
print(year_pd.to_string(index=False))

Completed + cancelled rows: 6,122,877 (filtered from 8,745,442 total)

TOP 10 OCCUPATIONS
                                                                         Occupation      count     pct
1                                                                      Not Provided  2,944,952  33.67%
2                               ELECTRICIAN (Alternate Title: Interior Electrician)    970,989  11.10%
3                                                                         CARPENTER    445,737   5.10%
4                                                                           PLUMBER    323,430   3.70%
5                                                        CONSTRUCTION CRAFT LABORER    302,619   3.46%
6                                                        PIPE FITTER (Construction)    260,321   2.98%
7                                                                SHEET METAL WORKER    162,380   1.86%
8   STRUCTURAL STEEL WORKER (Alternate Titles: Ironworker or Structural Ironworker)   

In [7]:
overall_wage_diff = (
    exiter_df
    .filter(
        pl.col("STARTING_WAGE").is_not_null() & pl.col("Exit_Wage").is_not_null()
    )
    .with_columns(
        (pl.col("Exit_Wage") - pl.col("STARTING_WAGE")).alias("wage_diff")
    )
    .select([
        pl.col("STARTING_WAGE").median().round(2).alias("median_starting_wage"),
        pl.col("Exit_Wage").median().round(2).alias("median_exit_wage"),
        pl.col("wage_diff").median().round(2).alias("median_wage_diff"),
        pl.len().alias("count"),
    ])
)

row = overall_wage_diff.to_pandas().iloc[0]
print(f"Count:              {int(row['count']):,}")
print(f"Median starting wage:  ${row['median_starting_wage']:,.2f}")
print(f"Median exit wage:      ${row['median_exit_wage']:,.2f}")
print(f"Median wage diff:      ${row['median_wage_diff']:,.2f}")

Count:              3,802,904
Median starting wage:  $15.78
Median exit wage:      $22.10
Median wage diff:      $6.20


In [8]:
wage_diff_by_year_age = (
    exiter_df
    .filter(
        pl.col("STARTING_WAGE").is_not_null() & pl.col("Exit_Wage").is_not_null()
    )
    .with_columns(
        (pl.col("Exit_Wage") - pl.col("STARTING_WAGE")).alias("wage_diff")
    )
    .group_by(["Fiscal Year", "Age Cohort"])
    .agg([
        pl.col("STARTING_WAGE").median().round(2).alias("median_starting_wage"),
        pl.col("Exit_Wage").median().round(2).alias("median_exit_wage"),
        pl.col("wage_diff").median().round(2).alias("median_wage_diff"),
        pl.len().alias("count"),
    ])
    .sort(["Fiscal Year", "Age Cohort"])
)

wage_pd = wage_diff_by_year_age.to_pandas()
for col in ["median_starting_wage", "median_exit_wage", "median_wage_diff"]:
    wage_pd[col] = wage_pd[col].apply(lambda x: f"${x:,.2f}")
wage_pd["count"] = wage_pd["count"].apply(lambda x: f"{x:,}")
print(wage_pd.to_string(index=False))

Fiscal Year                        Age Cohort median_starting_wage median_exit_wage median_wage_diff   count
       2014                 Ages 24 and Under               $14.22           $21.00            $7.00 116,409
       2014                        Ages 25-54               $15.00           $21.30            $5.70 202,975
       2014                          Ages 55+               $12.50           $14.44            $1.00   5,228
       2014 Participant Did Not Self-Identify               $10.00           $11.00            $1.00       1
       2015                 Ages 24 and Under               $14.59           $22.00            $7.00 133,203
       2015                        Ages 25-54               $15.02           $22.00            $6.14 240,265
       2015                          Ages 55+               $10.37           $14.53            $1.15   7,137
       2015 Participant Did Not Self-Identify               $24.03           $38.55           $14.52       3
       2016        

Summary statistics by year and state:

In [ ]:
# Pre-aggregate by fiscal year + state
agg = (
    df
    .with_columns([
        (pl.col("Apprentice Status Code").cast(pl.Utf8) == "CO").alias("is_co"),
        (pl.col("Apprentice Status Code").cast(pl.Utf8).is_in(["CO", "CA"])).alias("is_co_or_ca"),
        (pl.col("Gender").cast(pl.Utf8) == "Female").alias("is_female"),
        (pl.col("Gender").cast(pl.Utf8).is_in(["Male", "Female"])).alias("known_gender"),
        (pl.col("Veteran Status Title").cast(pl.Utf8).is_in(["Veteran", "Veteran, Eligible"])).alias("is_veteran"),
        (~pl.col("Veteran Status Title").cast(pl.Utf8).str.contains("Did Not")).alias("known_veteran"),
        (pl.col("Age Cohort").cast(pl.Utf8) == "Ages 24 and Under").alias("is_young"),
        (~pl.col("Age Cohort").cast(pl.Utf8).str.contains("Did Not")).alias("known_age"),
        (pl.col("Program View").cast(pl.Utf8) == "State").alias("is_state_program"),
    ])
    .group_by(["Fiscal Year", "APPR_STATE"])
    .agg([
        pl.len().alias("total"),
        pl.col("is_co").sum().alias("completed"),
        pl.col("is_co_or_ca").sum().alias("co_or_ca"),
        pl.col("STARTING_WAGE").median().alias("median_start_wage"),
        pl.col("Exit_Wage").median().alias("median_exit_wage"),
        pl.col("is_female").sum().alias("female_count"),
        pl.col("known_gender").sum().alias("known_gender"),
        pl.col("is_veteran").sum().alias("veteran_count"),
        pl.col("known_veteran").sum().alias("known_veteran"),
        pl.col("is_young").sum().alias("young_count"),
        pl.col("known_age").sum().alias("known_age"),
        pl.col("is_state_program").sum().alias("state_program_count"),
    ])
    .sort(["Fiscal Year", "APPR_STATE"])
)

df_pd = df.select(["Fiscal Year", "APPR_STATE", "Occupation"]).to_pandas()

top_occ_pd = (
    df_pd[
        df_pd["Occupation"].notna() &
        (df_pd["Occupation"] != "Not Provided")
    ]
    .groupby(["Fiscal Year", "APPR_STATE", "Occupation"])
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
    .groupby(["Fiscal Year", "APPR_STATE"])
    .apply(lambda g: g.head(5)[["Occupation", "count"]].to_dict(orient="records"), include_groups=False)
    .reset_index(name="top_occupations")
)

# Merge into the main aggregated dataframe
agg_pd = agg.to_pandas()
final_pd = agg_pd.merge(top_occ_pd, on=["Fiscal Year", "APPR_STATE"], how="left")
final_pd["top_occupations"] = final_pd["top_occupations"].apply(
    lambda x: x if isinstance(x, list) else []
)

# Export
records = final_pd.to_dict(orient="records")

with open("apprenticeship_agg.js", "w") as f:
    f.write("window._agg = ")
    json.dump(records, f, default=str)
    f.write(";")

print(f"Exported {len(records):,} rows")

# Sanity check
sample = [r for r in records if r["top_occupations"]]
print(f"Rows with occupations: {len(sample)}")
# print(sample[0]["top_occupations"])

Exported 628 rows
Rows with occupations: 617
[{'Occupation': 'ELECTRICIAN (Alternate Title: Interior Electrician)', 'count': 878}, {'Occupation': 'PLUMBER', 'count': 331}, {'Occupation': 'CARPENTER', 'count': 286}, {'Occupation': 'CONSTRUCTION CRAFT LABORER', 'count': 210}, {'Occupation': 'Operating Engineer', 'count': 172}]


# Pulling data using the DOL API

This code is just here for reference. The main analysis is done above with the downloaded zip file.

In [ ]:
# Import libraries
import requests, pandas as pd, json, warnings

warnings.filterwarnings("ignore")

with open("api_key.txt", "r") as file:
    API_KEY = file.read().strip()

st = pd.read_csv("state_abbr.csv", sep="\t")

Metadata:

In [ ]:
# def create_filter_condition(field, operator, value):
#     return {"and": [{"field": field, "operator": operator, "value": value}]}
def create_filter_condition(field, operator, value):
    condition = {"field": field, "operator": operator, "value": value}

    return json.dumps(condition)


params = {"X-API-KEY": API_KEY}

# Get metadata
metadata_url = "https://apiprod.dol.gov/v4/get/ETA/apprenticeship_data/json/metadata"
metadata_request = requests.get(metadata_url, params=params, verify=False)
metadata_json = metadata_request.json()
metadata = pd.DataFrame(metadata_json)
metadata

In [ ]:
# Build filters as dicts, not strings
filter_1 = create_filter_condition("union_y_n", "eq", "1")
filter_2 = create_filter_condition("value", "gt", "999")
filter_3 = create_filter_condition("industry", "in", ["A", "B", "C"])
filter_4 = create_filter_condition("industry", "like", "%A%")

url = "https://apiprod.dol.gov/v4/get/ETA/apprenticeship_data/json"

headers = {"X-API-KEY": API_KEY}

params = {
    "X-API-KEY": API_KEY,
    "limit": "1000",
    "offset": "0",
    # Pass the filter as a JSON string in the query param
    "filter_object": filter_1,  # Swap filter_1 with any of the other filters (2-4) to see different results.
}

request = requests.get(url, headers=headers, params=params, verify=False)
print(f"Request is {request}")
data_json = request.json()["data"]
data = pd.DataFrame(data_json)

Preview of the data:

In [ ]:
data.head()

Now we'll loop through each state and grab each state's data.

In [ ]:
st_list = sorted(st.abbrv.tolist())

In [ ]:
n = len(st_list)
i = 1

for s in st_list:
    print(f"On state {i} of {n}, {s}")
    filt = create_filter_condition("state", "eq", s)

    params = {
        "X-API-KEY": API_KEY,
        "limit": "1000",
        "offset": "0",
        # Pass the filter as a JSON string in the query param
        "filter_object": filt,  # Swap filter_1 with any of the other filters (2-4) to see different results.
    }

    request = requests.get(url, headers=headers, params=params, verify=False)
    print(f"Request is {request}")
    data_json = request.json()["data"]
    data = pd.DataFrame(data_json)

    if s == "AK":
        df = data
    else:
        df = pd.concat([df, data], ignore_index=True)

    i += 1

In [ ]:
df